In [1]:
import os
import re
import random
from glob import glob
import torch
import numpy as np
from datasets import load_dataset, Dataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors
from transformers import (
    TrainerCallback,
    PreTrainedTokenizerFast,
    LlamaConfig,
    LlamaForCausalLM,
    AutoTokenizer,
    AutoModelForCausalLM,
)

from trl import SFTTrainer, SFTConfig

import warnings
warnings.filterwarnings("ignore", message=".*IProgress not found.*")


In [2]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(42)

### Предобработка данных

In [5]:
def preprocess_texts(folder_path, output_file, chunk_size):

    # Загружаем все строки из всех файлов
    all_lines = []
    for file_path in glob(os.path.join(folder_path, "*.txt")):
        with open(file_path, 'r', encoding='utf-8') as f:
            all_lines.extend(f.readlines())

    # Удаляем дубликаты строк
    seen = set()
    unique_lines = []
    for line in all_lines:
        line_stripped = line.strip()
        if line_stripped and line_stripped not in seen:
            seen.add(line_stripped)
            unique_lines.append(line)
        elif not line_stripped:
            unique_lines.append(line)

    # Оставляем только кириллицу, цифры, пробелы, базовую пунктуацию
    cleaned_lines = []
    for line in unique_lines:
        clean = re.sub(r'[^а-яА-ЯёЁ0-9\s\.\,\!\?\;\:\(\)\"\'\-\–\—]', '', line)
        # убираем лишние пробелы
        clean = re.sub(r'\s+', ' ', clean).strip()
        if clean:
            cleaned_lines.append(clean)

    # Склеиваем
    full_text = ' '.join(cleaned_lines)

    # Нормализуем повторяющуюся пунктуацию
    full_text = re.sub(r'\.{2,}', '.', full_text)
    full_text = re.sub(r'!{2,}', '!', full_text)
    full_text = re.sub(r'\?{2,}', '?', full_text)
    full_text = re.sub(r',{2,}', ',', full_text)

    # Разбиваем на чанки фиксированной длины
    chunks = []
    for i in range(0, len(full_text), chunk_size):
        chunk = full_text[i:i+chunk_size]
        if len(chunk) > 50:  # отбрасываем совсем короткие куски
            chunks.append(f"<bos> {chunk} <eos>")

    # Сохраняем чанки в файл
    with open(output_file, 'w', encoding='utf-8') as f:
        for chunk in chunks:
            f.write(chunk + '\n')

    return chunks

In [6]:
# Обработаем текст
texts = preprocess_texts("./RussianNovels/corpus", "./dataset/preprocessed_chunks.txt", chunk_size=1000)

In [7]:
# Посмотрим на пример
texts[0]

'<bos> ГЛАВА 1 Иван Акимович Самгин любил оригинальное; поэтому, когда жена родила второго сына, Самгин, сидя у постели роженицы, стал убеждать ее: - Знаешь что. Вера, дадим ему какое-нибудь редкое имя? Надоели эти бесчисленные Иваны, Василии. А? Утомленная муками родов, Вера Петровна не ответила. Муж на минуту задумался, устремив голубиные глаза свои в окно, в небеса, где облака, изорванные ветром, напоминали и ледоход на реке и мохнатые кочки болота. Затем Самгин начал озабоченно перечислять, пронзая воздух коротеньким и пухлым пальцем: - Христофор? Кирик? Вукол? Никодим? Каждое имя он уничтожал вычеркивающим жестом, а перебрав десятка полтора необычных имен, воскликнул удовлетворенно: - Самсон! Самсон Самгин, - вот! Это не плохо! Имя библейского героя, а фамилия, - фамилия у меня своеобразная! - Не тряси кровать, - тихо попросила жена. Он извинился, поцеловал ее руку, обессиленную и странно тяжелую, улыбаясь, послушал злой свист осеннего ветра, жалобный писк ребенка. - Да, Самсон! Н

### Токенизатор

In [8]:
# Создаём BPE токенизатор
tokenizer = Tokenizer(models.BPE())

# Предтокенизация
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)

# Тренер с базовыми спецтокенами
trainer = trainers.BpeTrainer(vocab_size=3000, special_tokens=["<unk>", "<bos>", "<eos>", "<pad>"])

In [9]:
# Обучаем на созданном файле
tokenizer.train(["dataset/preprocessed_chunks.txt"], trainer=trainer)

In [10]:
# Добавляем постпроцессинг и декодер
tokenizer.post_processor = processors.ByteLevel(trim_offsets=False)
tokenizer.decoder = decoders.ByteLevel()

# Сохраняем
tokenizer.save("models/bpe_tokenizer.json")

In [11]:
# Проверим размер словаря
print(f"Размер словаря: {tokenizer.get_vocab_size()}")

Размер словаря: 3000


### Токенизация

In [12]:
# Загружаем обученный BPE-токенизатор
tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="models/bpe_tokenizer.json",
    bos_token="<bos>",
    eos_token="<eos>",
    pad_token="<pad>",
    unk_token="<unk>"
)

In [13]:
# Токенизируем с длиной контекста 512 токенов
def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

# Создаём Dataset из словаря
dataset = Dataset.from_dict({"text": texts})

# Применяем токенизацию ко всему датасету
tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_dataset = tokenized_dataset.map(lambda x: {"labels": x["input_ids"]})

Map:   0%|          | 0/43008 [00:00<?, ? examples/s]

Map:   0%|          | 0/43008 [00:00<?, ? examples/s]

In [14]:
# Преобразуем в формат PyTorch
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# Проверим результат
print(f"Размер датасета: {len(tokenized_dataset)}")
print(f"Пример input_ids: {tokenized_dataset[0]['input_ids'][:50]}...")

Размер датасета: 43008
Пример input_ids: tensor([   1,  371,   61,   93, 1575, 1517, 1575, 1206, 1266,  260, 1435, 1994,
        1044, 2984, 1443,  426,  180,  364,  412,   23, 2577,  104,  353,    9,
         493, 2625,  987,  340, 1789,  205, 2658,    9, 1044,    9,  775,  113,
         177, 1159,  445,  159,  213,  135, 1392,    9, 1066,  728, 1245,  209,
         347,   22])...


### Обучение SFT

In [15]:
print("CUDA доступен:", torch.cuda.is_available())
print("Имя GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA доступен: True
Имя GPU: Tesla T4


In [16]:
# Инициализируем модель Llama
config = LlamaConfig(
    hidden_size=1024,
    intermediate_size=1536,
    num_hidden_layers=16,
    num_attention_heads=16,
    num_key_value_heads=8,
    vocab_size=tokenizer.vocab_size,
    max_position_embeddings=512,
)
model = LlamaForCausalLM(config).cuda()
print(f"Количество параметров: {model.num_parameters():,}")

Количество параметров: 132,006,912


In [17]:
# Тестовые промпты
test_prompts = [
    "Все мысли, которые имеют огромные последствия",
    "Сила войска зависит от его духа",
    "Мысль о том, что он принес страдания",
    "Человек сознает себя свободным",
    "Что бы ни случилось, я всегда буду",
    "Любовь мешает смерти",
    "Нет, жизнь не кончена",
    "Всякая мысль, даже самая простая",
    "Война не любезность, а самое гадкое дело",
    "Чтобы жить честно"
]

In [20]:
# Коллбэк для генерации на тестовых промптах после каждой эпохи
class GenerateCallback(TrainerCallback):
    def __init__(self, tokenizer, prompts, max_new_tokens=50):
        self.tokenizer = tokenizer
        self.prompts = prompts
        self.max_new_tokens = max_new_tokens

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        if model is None:
            return
        model.eval()
        print(f"\n===== Генерация после эпохи {state.epoch} =====")
        for prompt in self.prompts:
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=self.max_new_tokens,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id
                )
            print(f"Промпт: {prompt}\nГенерация: {tokenizer.decode(outputs[0], skip_special_tokens=True)}\n")
        model.train()

# Аргументы обучения
sft_config = SFTConfig(
    output_dir="./models",
    learning_rate=1e-4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    max_length=256,
    num_train_epochs=3,
    report_to="none",
    logging_steps=500,
    weight_decay=0.01,
    fp16=True,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    remove_unused_columns=False,
    save_strategy="no",
    packing=True
)

# Trainer
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=tokenized_dataset,
    processing_class=tokenizer,
    callbacks=[GenerateCallback(tokenizer, test_prompts)],
)

# Запуск обучения
print("Training device:", trainer.model.device)
trainer.train()

Packing train dataset:   0%|          | 0/43008 [00:00<?, ? examples/s]

Training device: cuda:0


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/tr

Step,Training Loss
500,6.078443
1000,5.676803
1500,5.352962
2000,5.112943
2500,4.916168
3000,4.752690
3500,4.607747
4000,4.515458
4500,4.425464
5000,4.349104



===== Генерация после эпохи 1.0 =====
Промпт: Все мысли, которые имеют огромные последствия
Генерация: Все мысли, которые имеют огромные последствия, что они не знают, что они не знают, что они, как и они, не знают, что они, как они, не знают, что они, и что они, и что они, и они, и они

Промпт: Сила войска зависит от его духа
Генерация: Сила войска зависит от его духа, которой не только не влюблены в Вильну, но и в том, что в Москве, в Москве, в Москве, в Москве, в Москве, в Москве, в Москве, в Москв

Промпт: Мысль о том, что он принес страдания
Генерация: Мысль о том, что он принес страдания, и что он не верит в себе, он не верит, что он нехорошо, и что он нехорошо, и что он нехорошо, и что он нехорошо, он не может быть, он и не

Промпт: Человек сознает себя свободным
Генерация: Человек сознает себя свободным и, может быть, и не будет, и не будет, и не будет, и будет, и будет, и будет, и будет, и будет, и будет, и будет, и будет, и будет,,,,,,,

Промпт: Что бы ни случилось, я всегда

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:201: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids = [torch.tensor(ids) for ids in input_ids]
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:201: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids = [torch.tensor(ids) for ids in input_ids]
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:201: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids = [torch.tensor(ids) for ids in input_ids]
/usr/local/lib/python3.12/dist-


===== Генерация после эпохи 2.0 =====
Промпт: Все мысли, которые имеют огромные последствия
Генерация: Все мысли, которые имеют огромные последствия, и потому не может быть, чтобы они не были, как прежде, и не они, и те, которые, как они, были, и те, которые были, и те, которые были, и те, и те, и те,

Промпт: Сила войска зависит от его духа
Генерация: Сила войска зависит от его духа. Всякому, в том, чтобы не было, чтобы не было, чтобы не было, не было, не было, не было, не было, не было, не было, не было, что было, и, не, и

Промпт: Мысль о том, что он принес страдания
Генерация: Мысль о том, что он принес страдания, и что он не только не, но и не мог не быть, но и не мог бы быть. Он не мог не думать, и он не мог понять, что он может сделать. Он не мог понять, что он понимает, и

Промпт: Человек сознает себя свободным
Генерация: Человек сознает себя свободным, и, если б не,, то, что же делать? Или не хотят, чтобы не хотели, чтобы не хотели, и не будут, и не будут, и не будут, и не бу

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:201: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids = [torch.tensor(ids) for ids in input_ids]
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:202: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = [torch.tensor(lbl) for lbl in labels]
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:201: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids = [torch.tensor(ids) for ids in input_ids]
/usr/local/lib/python3.12/dist-packag


===== Генерация после эпохи 3.0 =====
Промпт: Все мысли, которые имеют огромные последствия
Генерация: Все мысли, которые имеют огромные последствия, и которые имеют право сказать, что они имеют право сказать, что они имеют право. -- Да, да, -- сказал Пьер, -- но они, как будто все они были в том, что они говорили, что

Промпт: Сила войска зависит от его духа
Генерация: Сила войска зависит от его духа. Сообразив это, Нехлюдов, не зная, что сказать, сказал: "Вот как же я скажу, что я не люблю, как я люблю, как я люблю". -- Да, да, -- сказал Нехлюдов, -- но

Промпт: Мысль о том, что он принес страдания
Генерация: Мысль о том, что он принес страдания, и что он не может быть иначе, как и он. Но он не понимал, что он не понимает того, что он говорил. И он, как всегда, в первый раз в жизни, в первый раз испытывал чувство,

Промпт: Человек сознает себя свободным
Генерация: Человек сознает себя свободным, и, не дыша, дыша, как бы вкрадчивый, в то, что он делал, он не мог не только не обнажить

TrainOutput(global_step=64512, training_loss=3.46293183818223, metrics={'train_runtime': 10662.5223, 'train_samples_per_second': 12.101, 'train_steps_per_second': 6.05, 'total_flos': 2.555243225992397e+16, 'train_loss': 3.46293183818223})

In [21]:
trainer.save_model("./models/pretrain_sft")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [22]:
def generate(model, tokenizer, prompt, max_new_tokens=100):
    model.eval()
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
    generated = input_ids
    with torch.no_grad():
        for _ in range(max_new_tokens):
            outputs = model(input_ids=generated)
            next_token_logits = outputs.logits[:, -1, :]
            next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
            generated = torch.cat((generated, next_token), dim=1)
            if next_token.item() == tokenizer.eos_token_id:
                break
    return tokenizer.decode(generated[0], skip_special_tokens=True)

for p in test_prompts:
    print(f"Prompt: {p}\nResponse:{generate(model, tokenizer, p)}\n")

Prompt: Все мысли, которые имеют огромные последствия
Response:Все мысли, которые имеют огромные последствия, и которые имеют право сказать, что они имеют право. -- Да, да, -- сказал Пьер, -- это то, что они имеют право сказать. -- Да, да, -- сказал Пьер, -- но я знаю, что это не-с. -- Да, да, -- сказал Пьер, -- но я знаю, что это не может быть, -- сказал Пьер. -- Я знаю, что это так, -- сказал Пьер. -- Я знаю, что это так

Prompt: Сила войска зависит от его духа
Response:Сила войска зависит от его духа. Сообразив это, Нехлюдов, не зная, что сказать, сказал: "Вот как же я скажу, что я не люблю, как я люблю, как я люблю". -- Да, да, -- сказал Нехлюдов, -- но я не понимаю, что я люблю, и я люблю, и люблю, и люблю, и люблю, и люблю, и люблю, и люблю. -- Да, да, да, -- сказал Нехлюдов, -- но я знаю, что это не я

Prompt: Мысль о том, что он принес страдания
Response:Мысль о том, что он принес страдания, и что он не может быть иначе, как и он. Но он не понимал, что он не понимает того, что 

## Post-train SFT

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B"          # базовая модель
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat32,
    device_map="auto"
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.chat_template = "{% for message in messages %}{% if message['role'] == 'user' %}{{ '<|im_start|>user\n' + message['content'] + '<|im_end|>\n' }}{% elif message['role'] == 'assistant' %}{{ '<|im_start|>assistant\n' + message['content'] + '<|im_end|>\n' }}{% endif %}{% endfor %}"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [24]:
ds = load_dataset("d0rj/alpaca-cleaned-ru", split="train")

README.md:   0%|          | 0.00/760 [00:00<?, ?B/s]

data/train-00000-of-00001-c503683bee003a(…):   0%|          | 0.00/36.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

In [25]:
def examples_to_messages(examples):
    data = {'messages': []}
    for example in examples:
        user_content = example['instruction']
        if example['input']:
            user_content += '\n' + example['input']
        data['messages'].append([
            {'role': 'user', 'content': user_content},
            {'role': 'assistant', 'content': example['output']}
        ])
    return Dataset.from_dict(data)

ds = examples_to_messages(ds)

In [26]:
def formatting_func(example):
    messages = example.get('messages', [])
    if not messages:
        return [""]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    if not isinstance(text, str):
        text = str(text)
    return [text]

In [32]:
sft_config = SFTConfig(
    output_dir="./models",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    learning_rate=2e-5,
    weight_decay=0.01,
    num_train_epochs=3,
    max_length=256,
    logging_steps=5000,
    save_strategy="no",
    report_to="none",
    dataset_text_field="text",
)

# Создание SFTTrainer
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=ds,
    processing_class=tokenizer,
    formatting_func=formatting_func,
)

# Запуск обучения
trainer.train()

Applying formatting function to train dataset:   0%|          | 0/51760 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/51760 [00:00<?, ? examples/s]

Step,Training Loss
1000,1.529805
2000,1.501881
3000,1.463036
4000,1.444745
5000,1.426003
6000,1.408867
7000,1.397350
8000,1.390548
9000,1.388577
10000,1.391141


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RuntimeError: [enforce fail at inline_container.cc:668] . unexpected pos 272349568 vs 272349456

In [ ]:
# Сохранение модели
trainer.save_model("./models/qwen_sft")

In [ ]:
questions = [
    "сколько планет в нашей солнечной системе?",
    "расскажи стих",
    "когда собирать крыжовник?",
    "Как быстро выучить новый язык?"
]

In [ ]:
def generate(model, tokenizer, prompt, max_new_tokens=100):
    model.eval()
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
    generated = input_ids
    with torch.no_grad():
        for _ in range(max_new_tokens):
            outputs = model(input_ids=generated)
            next_token_logits = outputs.logits[:, -1, :]
            next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
            generated = torch.cat((generated, next_token), dim=1)
            if next_token.item() == tokenizer.eos_token_id:
                break
    return tokenizer.decode(generated[0], skip_special_tokens=True)

for q in questions:
    print(f"Prompt: {q}\nResponce: {generate(model, tokenizer, q)}\n{'-'*50}")